# Distance from each high school to nearest UC and CSU

- **Paper:** (prep)
- **Original notebook:** `18_spatial_vars.ipynb`
- **Reads:**
  - `outputs/school_related.pkl`
  - `data/raw/CA_school_directory/Colleges_in_California.kml`
- **Writes:**
  - `outputs/distance_to_colleges.pkl`


In [ ]:
# Paths: edit config.py at the repo root, not this cell
import sys
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
sys.path.insert(0, str(_root))
from config import DATA_DIR, DERIVED_DIR, OUTPUT_DIR, FIG_DIR

In [ ]:
# setup

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

import fiona



In [ ]:
# import
fiona.drvsupport.supported_drivers["KML"] = 'rw'
school_vars = pd.read_pickle(f"{OUTPUT_DIR}/school_related.pkl")
univ  = gpd.read_file(f'{DATA_DIR}/CA_school_directory/Colleges_in_California.kml',driver="KML")

In [ ]:
csu = univ[univ["Name"].str.contains("State University")].reset_index()
# csu["type"] = "CSU"
uc = univ[univ["Name"].str.contains("University of California,")].reset_index()
# uc["type"] = "UC"
pub_univ = pd.concat([csu,uc])
uc = uc.to_crs("EPSG:3857")
csu = csu.to_crs("EPSG:3857")

pub_univ = pub_univ.to_crs("EPSG:3857")


In [ ]:
highschools = gpd.GeoDataFrame(school_vars,geometry = gpd.points_from_xy(school_vars["Longitude"],school_vars["Latitude"],crs = "EPSG:4326"))
highschools = highschools.to_crs("EPSG:3857")



In [ ]:
#highschools = highschools.drop(columns = ['distance_to_uc','distance_to_csu'])

highschools

In [ ]:
highschools_to_uc = gpd.sjoin_nearest(highschools,uc,distance_col="distance_to_uc",rsuffix = "UC",lsuffix = 'hs')
highschools_to_uc = highschools_to_uc.rename(columns = {"Name":"closest_UC"})

highschools_to_csu = gpd.sjoin_nearest(highschools,csu,distance_col="distance_to_csu",rsuffix = "CSU")
highschools_to_csu = highschools_to_csu.rename(columns = {"Name":"closest_CSU"})

highschools_to_uc = highschools_to_uc[['SchoolName',"CountyName",'closest_UC','distance_to_uc']]
highschools_to_csu = highschools_to_csu[['SchoolName',"CountyName",'closest_CSU','distance_to_csu']]

highschools = pd.merge(highschools,highschools_to_uc,on = ['SchoolName','CountyName'])

highschools = pd.merge(highschools,highschools_to_csu,on = ['SchoolName','CountyName'])

# with open(f'{OUTPUT_DIR}/school_related.pkl','wb') as f:
#   pickle.dump(highschools,f)


In [ ]:
highschools.columns

In [ ]:
highschools_to_college = pd.merge(highschools_to_uc,highschools_to_csu,on = ['SchoolName','CountyName'])
with open(f'{OUTPUT_DIR}/distance_to_colleges.pkl','wb') as f:
  pickle.dump(highschools_to_college,f)


In [ ]:
relevant = highschools[['college_going_rate','AP_Count',
            'YearsTeaching', 'years_teaching_std',
       'ChronicAbsenteeismRate', 'pct_frpm',
       'pct_passed_science', 'pct_passed_math', 'pct_passed_english',
       'school_expenditures','distance_to_uc',"distance_to_csu"]]
relevant = relevant.apply(pd.to_numeric, errors='coerce')



In [ ]:
sns.pairplot(relevant)
plt.show()


In [ ]:
cor_mat = relevant.corr(numeric_only = True)
matrix = np.triu(cor_mat)

plt.figure(figsize=(12, 8))
sns.heatmap(cor_mat, annot=True, cmap='coolwarm', fmt=".2f",mask = matrix)
plt.title('Correlation Matrix')
plt.show()